# FIREWORK CHRONOS — real-data pipeline in Google Colab
Runs in your browser (free). Downloads 25 years of NASA FIRMS hot spots, Earth Engine exposure/weather/vegetation, Open-Meteo daily weather, builds the calendar tables and trains the models.

**Keys:** typed with `getpass` and kept only in this Colab session's .env — never saved in the notebook. Do not share a notebook whose outputs show a key.

In [ ]:
!pip -q install earthengine-api httpx pandas pyarrow python-dotenv psycopg[binary] scikit-learn==1.9.1 joblib

## 1. Get the code
Upload `FIREWORK_pipeline.zip` (from the package) with the file panel on the left, or clone your GitHub repo.

In [ ]:
import os, zipfile
if os.path.exists('FIREWORK_pipeline.zip'):
    zipfile.ZipFile('FIREWORK_pipeline.zip').extractall('/content/fw')
# or: !git clone https://github.com/Dhruvsharma-DSAS/<your-repo>.git /content/fw
%cd /content/fw/pipeline
!ls

## 2. Keys (session only)

In [ ]:
from getpass import getpass
key = getpass('FIRMS MAP_KEY: ')
project = input('Earth Engine project id: ')
db = getpass('Supabase DATABASE_URL (Enter to skip): ')
with open('.env', 'w') as f:
    f.write(f'FIRMS_MAP_KEY={key}\nGEE_PROJECT={project}\nDATABASE_URL={db}\nFIRMS_MAX_TRANSACTIONS_PER_10MIN=4000\n')
del key, db
print('saved .env for this session only')

## 3. Earth Engine login (opens a Google sign-in)

In [ ]:
import ee
ee.Authenticate()
ee.Initialize(project=project)
print(ee.ImageCollection('MODIS/061/MOD14A1').filterDate('2024-11-01','2024-11-03').size().getInfo(), 'images — Earth Engine works')

## 4. Check everything

In [ ]:
!python s01_check_access.py

## 5. Quick test: one region, one season
If this works, run the full download.

In [ ]:
!python s02_firms_download.py --regions punjab --start 2019-10-01 --end 2019-11-30
!python s05_build_tables.py --regions punjab

## 6. Full run (resumable — rerun this cell if Colab disconnects)
Choose regions from `regions.json`. Punjab–Haryana ≈ 1,800 FIRMS requests.

In [ ]:
REGIONS = 'punjab_haryana cordoba_ar california new_south_wales'
!python run_all.py --regions $REGIONS --steps 2 3s 3 4 5 6

## 7. Save results to Google Drive (Colab files are deleted when the session ends)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/FIREWORK_data && cp -r data /content/drive/MyDrive/FIREWORK_data/
!cd data && zip -qr /content/drive/MyDrive/FIREWORK_data/FIREWORK_real_data.zip processed supabase_csv raw/firms/manifest.csv
print('saved to Drive: FIREWORK_data/')

## 8. Load into Supabase (run sql/schema_v3.sql in the Supabase SQL editor first)

In [ ]:
!python s07_load_supabase.py --regions $REGIONS

## 9. Train the models on the real data and read the model cards

In [ ]:
!python s08_train_models.py --regions $REGIONS
!head -60 ../ml/models/forecast.card.json